# Topic 36 — Attention
### Theory → Query/Key/Value intuition → scaled dot-product attention from scratch (NumPy) → PyTorch → visualize.

LSTMs (Topic 34) compress an entire sequence into ONE final hidden state — a bottleneck. Long
sequences lose detail this way. **Attention** lets the model look back at ALL positions in the
sequence directly and decide, for each output, how much to "attend to" (weight) each input position.
This is the conceptual bridge to Transformers (Topic 37).

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
torch.manual_seed(42)

## 1. Query, Key, Value — the intuition

Think of it like a search engine:
- **Query (Q)**: "what am I looking for?" — one per output position.
- **Key (K)**: "what does each input position offer?" — one per input position.
- **Value (V)**: "the actual content to retrieve" from each input position, if it matches.

For each query, compare it against every key (via dot product) to get a relevance SCORE per input
position, turn those scores into weights (via softmax), then take a weighted sum of the VALUES
using those weights. Positions that matched well contribute more.

## 2. Scaled dot-product attention — the formula

```text
Attention(Q, K, V) = softmax( Q @ K^T / sqrt(d_k) ) @ V
```

- `Q @ K^T`: raw similarity scores between every query and every key.
- `/ sqrt(d_k)`: scaling factor (d_k = key dimension) — keeps scores from growing too large as
  dimensionality increases, which would otherwise push softmax into regions with tiny gradients.
- `softmax(...)`: turns scores into attention WEIGHTS that sum to 1 across each query's row.
- `@ V`: weighted sum of values, using those attention weights.

In [ ]:
def softmax(x, axis=-1):
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    d_k = K.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)      # (n_queries, n_keys)
    weights = softmax(scores, axis=-1)    # each row sums to 1
    output = weights @ V                   # (n_queries, d_v)
    return output, weights

# Toy example: a 4-word "sentence", each word has a 3-dim representation
seq_len, d_model = 4, 3
X = rng.normal(0, 1, size=(seq_len, d_model))

# In self-attention, Q, K, V all come from the SAME input, via learned projections
W_q = rng.normal(0, 0.5, size=(d_model, d_model))
W_k = rng.normal(0, 0.5, size=(d_model, d_model))
W_v = rng.normal(0, 0.5, size=(d_model, d_model))

Q, K, V = X @ W_q, X @ W_k, X @ W_v

output, attention_weights = scaled_dot_product_attention(Q, K, V)
print("attention weights (each row = one word's attention distribution over all 4 words):")
print(np.round(attention_weights, 3))
print("\nrow sums (should all be 1.0):", attention_weights.sum(axis=1))
print("\noutput shape:", output.shape, "(same shape as input -- a re-weighted combination of all words)")

## 3. Visualizing attention weights

A heatmap of attention weights shows, for each word (row), how much it "attends to" every other
word (column) — this is the actual interpretability tool used to inspect what Transformers learned.

In [ ]:
words = ["you", "are", "so", "stupid"]

plt.figure(figsize=(5, 4))
plt.imshow(attention_weights, cmap="Blues")
plt.xticks(range(len(words)), words)
plt.yticks(range(len(words)), words)
plt.xlabel("attending TO"); plt.ylabel("attending FROM")
plt.colorbar(label="attention weight")
plt.title("Attention weights heatmap")
for i in range(len(words)):
    for j in range(len(words)):
        plt.text(j, i, f"{attention_weights[i,j]:.2f}", ha="center", va="center", fontsize=8)
plt.show()
# Each ROW sums to 1 -- it's a probability distribution over "how much this word looks at every word".
# With random untrained weights this pattern is meaningless; a TRAINED model learns weights where,
# e.g., "stupid" attends strongly to "you" (the target of the insult).

## 4. Self-attention: why it's called "self"

**Self-attention** = attention where Q, K, AND V all come from the same sequence — each word
attends to every other word IN THE SAME SENTENCE (including itself), as opposed to attending to a
different sequence entirely (e.g. in translation, attending from the output language back to the
input language — "cross-attention", used in Transformer decoders).

In [ ]:
# Confirm: word 0 ("you") CAN attend to itself
print("'you' attending to itself:", round(attention_weights[0, 0], 3))
print("'you' attending to 'stupid':", round(attention_weights[0, 3], 3))

## 5. Multi-head attention preview (full treatment in Topic 37)

Instead of computing attention ONCE, split Q/K/V into several smaller "heads", compute attention
independently in each, then concatenate the results. Different heads can learn to focus on
different kinds of relationships (e.g. one head tracks negation, another tracks subject-object
pairs) simultaneously.

In [ ]:
def multi_head_attention_scratch(X, n_heads, d_model):
    d_head = d_model // n_heads
    head_outputs = []
    for head in range(n_heads):
        Wq = rng.normal(0, 0.5, (d_model, d_head))
        Wk = rng.normal(0, 0.5, (d_model, d_head))
        Wv = rng.normal(0, 0.5, (d_model, d_head))
        Qh, Kh, Vh = X @ Wq, X @ Wk, X @ Wv
        out, _ = scaled_dot_product_attention(Qh, Kh, Vh)
        head_outputs.append(out)
    return np.concatenate(head_outputs, axis=-1)   # concatenate all heads back together

mha_output = multi_head_attention_scratch(X, n_heads=3, d_model=6 if d_model < 6 else d_model)
print("multi-head output shape:", mha_output.shape)

## 6. PyTorch's built-in attention: `nn.MultiheadAttention`

In [ ]:
mha = nn.MultiheadAttention(embed_dim=8, num_heads=2, batch_first=True)

X_torch = torch.randn(1, 4, 8)   # (batch, seq_len, embed_dim)
attn_output, attn_weights = mha(X_torch, X_torch, X_torch)   # self-attention: Q=K=V=X_torch

print("output shape:", attn_output.shape)
print("attention weights shape:", attn_weights.shape, "-> (batch, seq_len, seq_len)")

plt.figure(figsize=(4, 3.5))
plt.imshow(attn_weights[0].detach().numpy(), cmap="Blues")
plt.colorbar()
plt.title("PyTorch MultiheadAttention weights (untrained, random)")
plt.show()

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Manually construct a Q, K, V for a 3-word sentence where you EXPECT word 2 to attend
#    strongly to word 0 (design the vectors so their dot product is large) -- confirm the
#    resulting attention weight is indeed high.
# 2. Change n_heads in multi_head_attention_scratch to 1 and to 6 -- how does d_head
#    (dimension per head) change, and what constraint must n_heads satisfy relative to d_model?
# 3. Increase seq_len to 10 in the scratch example and re-plot the attention heatmap --
#    does it get harder to read as sequences grow?
# 4. In one sentence: why can attention (unlike an LSTM's single final hidden state) let a model
#    directly connect a word to something 50 words earlier without any information loss?

---
### Next up: **Topic 37 — Transformers** (self-attention + positional encoding + feed-forward, assembled into the full architecture).

Say "next" when you're ready.